# Project: Drift Detective 🔍
A model trained on last year's electricity market slowly goes stale. Play **ML on-call**: train a DK1 price model on Oct 2023 – Jun 2024, then watch it month by month. Measure **data drift** (PSI on the inputs), **performance decay** (MAE), raise alerts, and show that monthly retraining pays off.

Topic: [[MLOps Overview]], [[Model Deployment and Serving]] · guide note: [[Project - Drift Detective]] · data as in [[Danish Electricity Prices]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import json, pathlib, urllib.parse, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

def energinet(ds, start="2023-10-01", end="2025-10-01"):
    f = pathlib.Path(f"data/{ds}_DK1_{start}_{end}.csv"); f.parent.mkdir(exist_ok=True)
    if not f.exists():
        q = {"start": start, "end": end, "filter": json.dumps({"PriceArea": ["DK1"]}), "limit": 0}
        pd.DataFrame(json.load(urllib.request.urlopen("https://api.energidataservice.dk/dataset/" + ds + "?" + urllib.parse.urlencode(q)))["records"]).to_csv(f, index=False)
    return pd.read_csv(f)
p = energinet("Elspotprices"); fc = energinet("Forecasts_Hour")
df = pd.DataFrame({"y": p.assign(t=pd.to_datetime(p.HourUTC)).set_index("t").SpotPriceEUR.sort_index()}).asfreq("h")
wide = fc.assign(t=pd.to_datetime(fc.HourUTC)).pivot_table(index="t", columns="ForecastType", values="ForecastDayAhead").reindex(df.index)
local = df.index.tz_localize("UTC").tz_convert("Europe/Copenhagen")
df["hour"], df["dow"] = local.hour, local.dayofweek
df["wind"] = wide[["Offshore Wind", "Onshore Wind"]].sum(1, min_count=1); df["solar"] = wide["Solar"]
for L in (24, 48, 168): df[f"lag{L}"] = df.y.shift(L)
df = df.dropna(); FEATS = ["hour", "dow", "wind", "solar", "lag24", "lag48", "lag168"]
train = df[df.index < "2024-07-01"]; live = df[df.index >= "2024-07-01"]
model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0).fit(train[FEATS], train.y)
print(f"trained on {len(train):,} hours; live period {live.index.min():%Y-%m} → {live.index.max():%Y-%m}")

## 1. Population Stability Index
Bin the **training** distribution of a feature into deciles, then compare the share of live data in each bin: $\text{PSI} = \sum_b (a_b - e_b)\ln\frac{a_b}{e_b}$. Rule of thumb: < 0.1 stable, 0.1–0.25 moderate, > 0.25 major shift. Clip shares at `eps` to avoid $\log 0$.

In [ ]:
def psi(expected, actual, bins=10, eps=1e-4):
    # TODO 1: quantile bin edges from `expected` (outer edges ±inf), shares per bin, clip, PSI
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_psi():
    r = np.random.default_rng(0); a = r.normal(0, 1, 20000)
    same, shifted = psi(a, r.normal(0, 1, 20000)), psi(a, r.normal(1, 1, 20000))
    return same < 0.01 and shifted > 0.25 and psi(np.arange(10.0), np.arange(10.0), bins=5) < 1e-9
check("PSI", _t_psi, "Same distribution → PSI ≈ 0; a 1-σ mean shift → PSI > 0.25.")

In [ ]:
mae = lambda a, b: float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))
base = mae(train.y.iloc[-24 * 60:], model.predict(train[FEATS].iloc[-24 * 60:]))    # in-sample MAE of the last 60 days (optimistic)
if ready("PSI"):
    rows = {}
    for month, g in live.groupby(live.index.to_period("M")):
        drift = {f: psi(train[f], g[f]) for f in ["wind", "solar", "lag24"]}
        rows[str(month)] = dict(**{f"psi_{k}": v for k, v in drift.items()}, psi_max=max(drift.values()), mae=mae(g.y, model.predict(g[FEATS])))
    report = pd.DataFrame(rows).T
    print(report.round(3).to_string())
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.2)); report[["psi_wind", "psi_solar", "psi_lag24"]].plot(ax=ax[0], marker="o", title="input drift (PSI)")
    ax[0].axhline(0.25, ls="--", c="red"); report.mae.plot(ax=ax[1], marker="o", title="monthly MAE (EUR/MWh)"); plt.show()

## 2. Alerting rule
Return the months that should page you: any input PSI above `psi_limit`, **or** MAE more than `mae_factor` × the baseline MAE.

In [ ]:
def alerts(report, baseline_mae, psi_limit=0.25, mae_factor=1.5):
    # TODO 2: list of month labels where psi_max > psi_limit or mae > mae_factor * baseline_mae
    raise NotImplementedError  # TODO 2

In [ ]:
check("alerts", lambda: alerts(pd.DataFrame({"psi_max": [0.1, 0.3, 0.1], "mae": [10, 10, 40]}, index=["a", "b", "c"]), 20) == ["b", "c"],
      "Month b drifts (PSI 0.3), month c decays (MAE 40 > 1.5·20).")
if ready("PSI", "alerts"):
    print("🚨 alerts:", alerts(report, base))
    print("→ Almost every month alerts: wind and solar are SEASONAL, so comparing July with an autumn–winter training set always 'drifts'. Alert fatigue is a real MLOps problem; see the first stretch goal for a fix.")
    static, retrained = [], []
    for month, g in live.groupby(live.index.to_period("M")):
        hist = df[df.index < g.index.min()]                                  # everything known before this month
        m2 = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0).fit(hist[FEATS], hist.y)
        static.append(mae(g.y, model.predict(g[FEATS]))); retrained.append(mae(g.y, m2.predict(g[FEATS])))
    print(f"live MAE – frozen model {np.mean(static):.2f}, retrained monthly {np.mean(retrained):.2f} EUR/MWh")
    check("retraining pays off", lambda: np.mean(retrained) < np.mean(static), "Monthly retraining should beat the frozen model on average.")

<details><summary>▶ Solution</summary>

```python
def psi(expected, actual, bins=10, eps=1e-4):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected)
    a = np.histogram(actual, edges)[0] / len(actual)
    e, a = np.clip(e, eps, None), np.clip(a, eps, None)
    return float(np.sum((a - e) * np.log(a / e)))
```

```python
def alerts(report, baseline_mae, psi_limit=0.25, mae_factor=1.5):
    return [m for m, row in report.iterrows() if row.psi_max > psi_limit or row.mae > mae_factor * baseline_mae]
```
</details>

## Stretch goals
- Seasonal drift is *expected* (solar in summer). Compare each month with the **same month last year** instead of the whole training set.
- Use the KS test (`scipy.stats.ks_2samp`): with 700 hours per month, why is everything "significant"?
- Label delay: in real life you learn the true price a day later; for credit default it's years. How would you monitor then?